# Assignment 4 — EvaLoop Telugu-summary loop: run everything

**Before you start:** add a Colab secret named `GEMINI_API_KEY` (🔑 icon in the left sidebar, turn on *Notebook access*).
The key must belong to a Google Cloud project with **billing on** (the free tier is too slow).

Then **Runtime → Run all**. Results are written to Google Drive (`MyDrive/a4_runs`), so if Colab disconnects,
just *Run all* again: finished runs are skipped and every answer already received is replayed from the cache.

Generated code is untrusted model output; it is executed only here, inside this Colab VM, by EvalPlus.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/a4_runs'
os.makedirs(OUT, exist_ok=True)
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
print('key loaded:', bool(os.environ['GEMINI_API_KEY']))

In [ ]:
%cd /content
!rm -rf EvaLooop
!git clone -q -b a4-telugu-loop https://github.com/IG-08/EvaLooop.git
%cd /content/EvaLooop
!git log --oneline -1
!pip install -q evalplus openai fire tqdm pytest scipy
!pip install -q -e . --no-deps

## 1. Tests (should end with `100 passed`)

In [ ]:
!python -m pytest -q 2>&1 | tail -3

## 2. Preflight: two real calls per model
Each model should print two `[ok]` lines (one Python function, one Telugu sentence) and a cost estimate.
If you see `[FAIL]`, stop and send a screenshot.

In [ ]:
!python a4/preflight.py --models gemini-2.5-flash-lite,gemini-3.1-flash-lite:minimal

## 3. The experiment (about 1.5–2 hours, mostly waiting)
2 models × 4 configurations on all 378 MBPP+ tasks, temperature 0, max 10 cycles.
Each finished run prints a `[done]` line. Keep this tab open.

In [ ]:
!python a4/run_experiments.py --out "$OUT" --models gemini-2.5-flash-lite,gemini-3.1-flash-lite:minimal

## 4. Summary tables and the failed-loop sample

In [ ]:
!python a4/summarize.py --results "$OUT" --out "$OUT/analysis"
!python a4/failures.py --results "$OUT" --out "$OUT/analysis/inspection" --per_cell 8

## 5. Download `a4_results.zip` and attach it in the chat

In [ ]:
import shutil, glob, zipfile
from google.colab import files
zip_path = '/content/a4_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f'{OUT}/**/*', recursive=True):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, OUT))
print(round(os.path.getsize(zip_path) / 1e6, 1), 'MB')
files.download(zip_path)